# 03 — Surgical-depth scaling and architecture selection

This notebook turns the dipole scaling laws into a first hardware-screening calculation. It does **not** replace a finite-magnet model or measurements.

We compare the on-axis force and maximum torque of example EPM/IPM moments over clinically relevant separations, then build a transparent architecture scorecard.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.magnetics import dipole_field, dipole_force


## Example moments and depth sweep

The source and receiver moments below are placeholders. Replace them with identified values and repeat with measured field/force maps. For axial attraction both moments and the separation axis are aligned. For maximum torque the IPM is perpendicular to the local field.


In [ ]:
m_E_mag = 25.0      # A m^2, illustrative large external permanent magnet
m_I_mag = 0.050     # A m^2, illustrative embedded magnet
depths_mm = np.linspace(50, 250, 401)

forces, torques = [], []
for depth_mm in depths_mm:
    r = np.array([0.0, 0.0, depth_mm/1000])
    m_E = np.array([0.0, 0.0, m_E_mag])
    m_I_axial = np.array([0.0, 0.0, m_I_mag])
    m_I_perp = np.array([m_I_mag, 0.0, 0.0])
    forces.append(abs(dipole_force(r, m_E, m_I_axial)[2]))
    torques.append(m_I_mag*np.linalg.norm(dipole_field(r, m_E)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].semilogy(depths_mm, forces, lw=2.5)
axes[0].set(xlabel="separation [mm]", ylabel="axial attraction [N]", title="Illustrative force")
axes[1].semilogy(depths_mm, torques, lw=2.5, color="tab:orange")
axes[1].set(xlabel="separation [mm]", ylabel="maximum torque [N m]", title="Illustrative torque")
for ax in axes:
    ax.grid(True, which="both", alpha=.3)
plt.tight_layout()
plt.show()


In [ ]:
query_depths = np.array([75, 100, 125, 150, 180, 200])
table = pd.DataFrame({"depth_mm": query_depths})
table["force_N"] = np.interp(query_depths, depths_mm, forces)
table["max_torque_Nm"] = np.interp(query_depths, depths_mm, torques)
table


## Architecture scorecard

Scores are deliberately explicit and editable: 1 is poor, 5 is strong. The example weights reflect a human-scale minimally invasive system that values deep reach, clinical access, fast control, and thermal safety. They are **not universal rankings**.


In [ ]:
scores = pd.DataFrame({
    "deep_reach":       [4, 4, 3, 3, 5, 3],
    "fast_modulation":  [2, 2, 3, 5, 4, 1],
    "patient_access":   [3, 2, 3, 2, 5, 4],
    "low_heat":         [5, 5, 5, 2, 2, 5],
    "field_flexibility": [2, 4, 4, 5, 4, 1],
    "simple_calibration":[3, 2, 2, 3, 2, 3],
}, index=[
    "single robot EPM", "dual robot EPM", "rotating-PM array",
    "fixed coil array", "under-table/mobile coils", "magnetic anchor"
])
weights = pd.Series({
    "deep_reach": .24, "fast_modulation": .16, "patient_access": .22,
    "low_heat": .12, "field_flexibility": .18, "simple_calibration": .08
})
scores["weighted_total"] = scores[weights.index].mul(weights).sum(axis=1)
scores.sort_values("weighted_total", ascending=False)


In [ ]:
ordered = scores.sort_values("weighted_total")
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.barh(ordered.index, ordered["weighted_total"], color="tab:blue")
ax.set(xlim=(0, 5), xlabel="weighted score [1–5]",
       title="Example decision score — replace with project evidence")
ax.grid(axis="x", alpha=.3)
plt.tight_layout()
plt.show()


## How to use this responsibly

1. Replace moment estimates with identified magnet parameters.
2. Replace the dipole sweep with the calibrated or finite-magnet model inside its valid domain.
3. Define force/torque requirements from the task, flow, tether, gravity, and tissue contact.
4. Score architectures with measured workspace, heat, collision, imaging, and access data.
5. Perform sensitivity analysis over uncertain anatomy and preferences.
6. Keep “no feasible architecture yet” as a valid outcome.
